In [ ]:
import pandas as pd

customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
sellers = pd.read_csv("sellers.csv")
sales = pd.read_csv("sales_fact.csv")

print(sales.shape, customers.shape, products.shape, sellers.shape)

(50000, 12) (5000, 9) (10000, 6) (1000, 5)


In [ ]:
print(sales.dtypes)
print()
print(sales['Delivery_Status'].value_counts())
print()
print('Negative Total_Amount:', (sales['Total_Amount'] < 0).sum())
print('Duplicate Order_IDs:', sales['Order_ID'].duplicated().sum())

Order_ID            object
Customer_ID         object
Product_ID          object
Seller_ID           object
Quantity             int64
Order_Date          object
Shipping_Cost      float64
Discount_Amount    float64
Payment_Method      object
Total_Amount       float64
Delivery_Status     object
Review_Rating        int64
dtype: object

Delivery_Status
Delivered    16770
Pending      16746
Cancelled    16484
Name: count, dtype: int64

Negative Total_Amount: 708
Duplicate Order_IDs: 0


In [ ]:
sales['Order_Date'] = pd.to_datetime(sales['Order_Date'])
print(sales['Order_Date'].dtype)

datetime64[ns]


In [ ]:
before = len(sales)
sales_clean = sales[sales['Delivery_Status'] == 'Delivered'].copy()
print(f"Dropped {before - len(sales_clean):,} rows that were Pending or Cancelled")

Dropped 33,230 rows that were Pending or Cancelled


In [ ]:
print('Negative Total_Amount in delivered orders:', (sales_clean['Total_Amount'] < 0).sum())

Negative Total_Amount in delivered orders: 230


In [ ]:
neg = sales_clean[sales_clean['Total_Amount'] < 0]
print(neg[['Order_ID', 'Quantity', 'Discount_Amount', 'Shipping_Cost', 'Total_Amount']].head(10))

      Order_ID  Quantity  Discount_Amount  Shipping_Cost  Total_Amount
102    ORD_103         3           168.61          29.43        -91.30
291    ORD_292         1           171.18           8.82        -86.80
851    ORD_852         1           103.50          13.11        -79.38
904    ORD_905         1           140.33          23.70        -58.08
1117  ORD_1118         1           183.49          39.03        -69.31
1646  ORD_1647         3           169.19          10.11        -54.02
1750  ORD_1751         1           158.97          15.99        -45.05
1969  ORD_1970         1           158.00          29.98        -61.68
2098  ORD_2099         1            70.22           5.70        -25.23
2228  ORD_2229         1           160.40          34.67         -0.68


In [ ]:
before = len(sales_clean)
sales_clean = sales_clean[sales_clean['Total_Amount'] >= 0].copy()
print(f"Dropped {before - len(sales_clean):,} rows with negative Total_Amount")

# also check for missing values now, just to be thorough
print(sales_clean.isnull().sum())

Dropped 230 rows with negative Total_Amount
Order_ID           0
Customer_ID        0
Product_ID         0
Seller_ID          0
Quantity           0
Order_Date         0
Shipping_Cost      0
Discount_Amount    0
Payment_Method     0
Total_Amount       0
Delivery_Status    0
Review_Rating      0
dtype: int64


In [ ]:
sales_clean.to_csv("sales_clean.csv", index=False)

print(f"Final clean rows: {len(sales_clean):,}")
print(f"Unique customers: {sales_clean['Customer_ID'].nunique():,}")
print(f"Date range: {sales_clean['Order_Date'].min()} to {sales_clean['Order_Date'].max()}")
print(f"Total revenue: ₹{sales_clean['Total_Amount'].sum():,.2f}")

Final clean rows: 16,540
Unique customers: 4,812
Date range: 2023-01-01 00:00:00 to 2023-12-30 00:00:00
Total revenue: ₹49,422,492.40


In [ ]:
import numpy as np

snapshot_date = sales_clean['Order_Date'].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

rfm = sales_clean.groupby('Customer_ID').agg(
    Recency=('Order_Date', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Order_ID', 'count'),
    Monetary=('Total_Amount', 'sum')
).reset_index()

print(rfm.shape)
rfm.head()

Snapshot date: 2023-12-31 00:00:00
(4812, 4)


,Customer_ID,Recency,Frequency,Monetary
0,CUST_1,4,5,12055.52
1,CUST_10,122,2,6645.34
2,CUST_100,9,3,8193.39
3,CUST_1000,25,4,12777.02
4,CUST_1001,74,4,13082.95


In [ ]:
# Recency: LOWER is better, so we reverse the labels (5=most recent, 1=least recent)
r_quantiles = np.percentile(rfm['Recency'], [20, 40, 60, 80])
rfm['R_Score'] = 5 - np.digitize(rfm['Recency'], r_quantiles)

# Frequency: HIGHER is better
f_quantiles = np.percentile(rfm['Frequency'], [20, 40, 60, 80])
rfm['F_Score'] = 1 + np.digitize(rfm['Frequency'], f_quantiles)

# Monetary: HIGHER is better
m_quantiles = np.percentile(rfm['Monetary'], [20, 40, 60, 80])
rfm['M_Score'] = 1 + np.digitize(rfm['Monetary'], m_quantiles)

rfm[['Recency','R_Score','Frequency','F_Score','Monetary','M_Score']].head(10)

,Recency,R_Score,Frequency,F_Score,Monetary,M_Score
0,4,5,5,5,12055.52,4
1,122,2,2,2,6645.34,2
2,9,5,3,3,8193.39,3
3,25,4,4,4,12777.02,4
4,74,3,4,4,13082.95,4
5,11,5,5,5,24171.77,5
6,237,1,1,1,4232.38,2
7,4,5,4,4,19782.57,5
8,67,3,4,4,18803.46,5
9,26,4,6,5,17824.06,5


In [ ]:
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 4 and row['F_Score'] >= 3:
        return 'Loyal Customers'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New Customers'
    elif row['R_Score'] == 3:
        return 'Potential Loyalists'
    elif row['R_Score'] == 2:
        return 'At Risk'
    elif row['R_Score'] <= 1 and row['F_Score'] >= 3:
        return 'Cant Lose Them'
    else:
        return 'Churned'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['Segment'].value_counts()

,count
Segment,
At Risk,972
Potential Loyalists,963
Champions,836
Loyal Customers,732
Churned,659
New Customers,345
Cant Lose Them,305


In [ ]:
rfm.to_csv("rfm_scores.csv", index=False)

print(rfm['Segment'].value_counts())
print()
print("Revenue by segment:")
print(rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False))

Segment
At Risk                972
Potential Loyalists    963
Champions              836
Loyal Customers        732
Churned                659
New Customers          345
Cant Lose Them         305
Name: count, dtype: int64

Revenue by segment:
Segment
Champions              15509836.23
Potential Loyalists    10568721.86
At Risk                 9071993.44
Loyal Customers         6179328.40
Cant Lose Them          3278201.69
Churned                 3030373.39
New Customers           1784037.39
Name: Monetary, dtype: float64
